In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Choose model (recommended)
model_name = "microsoft/Phi-3-mini-4k-instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# IMPORTANT: QLoRA 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model loaded successfully in 4-bit mode")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model loaded successfully in 4-bit mode


In [5]:
model = prepare_model_for_kbit_training(model)

In [7]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA configuration (FIXED)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Show trainable params
model.print_trainable_parameters()

print("✅ LoRA applied successfully")

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327
✅ LoRA applied successfully


In [11]:
from datasets import load_dataset
from google.colab import files
import os

# -------------------------------
# STEP 1: Upload dataset files
# -------------------------------
print("📂 Upload train.jsonl and val.jsonl")
uploaded = files.upload()

# -------------------------------
# STEP 2: Verify files
# -------------------------------
print("📁 Current directory files:", os.listdir())

# -------------------------------
# STEP 3: Load dataset
# -------------------------------
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl"
})

print("✅ Dataset loaded")

# -------------------------------
# STEP 4: Tokenization
# -------------------------------
def tokenize_function(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""

    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_datasets = dataset.map(tokenize_function)

print("✅ Dataset tokenized and ready")

📂 Upload train.jsonl and val.jsonl


Saving train.jsonl to train (1).jsonl
Saving val.jsonl to val.jsonl
📁 Current directory files: ['.config', 'val.jsonl', 'train (1).jsonl', 'train.jsonl', 'sample_data']


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded


Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

✅ Dataset tokenized and ready


In [4]:
from google.colab import files
files.upload()

Saving train.jsonl to train.jsonl


{'train.jsonl': b'{"instruction": "Extract structured information", "input": "Name: Zoya Fatima, Age: 22, Role: ML Intern", "output": "{\\"name\\": \\"Zoya Fatima\\", \\"age\\": 22, \\"role\\": \\"ML Intern\\"}"}\n{"instruction": "Solve step by step", "input": "If a car travels 100 km in 2 hours, what is the speed?", "output": "Speed = Distance / Time = 100 / 2 = 50 km/h."}\n{"instruction": "Answer the question", "input": "What is a variable?", "output": "A variable is a container for storing data values."}\n{"instruction": "Extract structured information", "input": "Name: Zoya Fatima, Age: 22, Role: ML Intern", "output": "{\\"name\\": \\"Zoya Fatima\\", \\"age\\": 22, \\"role\\": \\"ML Intern\\"}"}\n{"instruction": "Extract structured information", "input": "Name: Zoya Fatima, Age: 22, Role: ML Intern", "output": "{\\"name\\": \\"Zoya Fatima\\", \\"age\\": 22, \\"role\\": \\"ML Intern\\"}"}\n{"instruction": "Answer the question", "input": "What is Python?", "output": "Python is a high

In [5]:
import os
print(os.listdir())

['.config', 'train.jsonl', 'sample_data']


In [6]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": "train.jsonl"})

dataset = dataset["train"].train_test_split(test_size=0.1)

print("✅ Dataset loaded successfully")

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded successfully


In [9]:
!pip install -U bitsandbytes accelerate peft transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model loaded with 4-bit quantization")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model loaded with 4-bit quantization


In [3]:
# -------------------------------
# IMPORTS
# -------------------------------
from datasets import load_dataset

# -------------------------------
# LOAD DATASET (SAFE)
# -------------------------------
try:
    dataset
    print("✅ Dataset already exists")
except:
    print("⚠️ Loading dataset...")
    dataset = load_dataset("json", data_files={"train": "train.jsonl"})
    dataset = dataset["train"].train_test_split(test_size=0.1)
    print("✅ Dataset loaded & split")

# -------------------------------
# CHECK TOKENIZER
# -------------------------------
try:
    tokenizer
    print("✅ Tokenizer already exists")
except:
    raise Exception("❌ Tokenizer not found. Run model loading cell first.")

# -------------------------------
# TOKENIZATION
# -------------------------------
def tokenize_function(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""

    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

print("🚀 Tokenizing dataset...")

tokenized_datasets = dataset.map(tokenize_function)

print("✅ Tokenization complete")

⚠️ Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded & split
✅ Tokenizer already exists
🚀 Tokenizing dataset...


Map:   0%|          | 0/972 [00:00<?, ? examples/s]

Map:   0%|          | 0/108 [00:00<?, ? examples/s]

✅ Tokenization complete


In [4]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

print("✅ LoRA applied")
model.print_trainable_parameters()

✅ LoRA applied
trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


In [5]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# detect split safely
if "test" in tokenized_datasets:
    eval_split = "test"
else:
    tokenized_datasets = tokenized_datasets["train"].train_test_split(test_size=0.1)
    eval_split = "test"

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets[eval_split],
    data_collator=data_collator
)

print("🚀 Training started...")

trainer.train()

print("🎉 Training completed")

🚀 Training started...


Step,Training Loss,Validation Loss
50,0.042144,0.043895
100,0.039237,0.037200
150,0.031522,0.031361
200,0.031319,0.031835
250,0.033767,0.030050
300,0.033835,0.032364
350,0.026200,0.031447
400,0.030961,0.029795
450,0.033194,0.029370
500,0.030661,0.030812


🎉 Training completed


In [6]:
model.save_pretrained("adapters")
tokenizer.save_pretrained("adapters")

print("✅ Adapter saved")

✅ Adapter saved
